# Practical 3 — Text Preprocessing & TF-IDF
**Name:** Paras Vishwakarma | **Roll No:** 48 | **Class:** C | **BE SEM-II 2025-26**

**Topics:** Text Cleaning · Lemmatization · Stop Word Removal · Label Encoding · TF-IDF

**Dataset:** News_dataset.pickle (PICT-NLP GitHub)

## Step 1 — Install & Import Libraries

In [1]:
!pip install nltk scikit-learn pandas numpy

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
import pickle
import re
import string
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

All libraries imported successfully!


## Step 2 — Load Dataset
Download `News_dataset.pickle` from:
https://github.com/PICT-NLP/BE-NLP-Elective/blob/main/3%20Preprocessing/News_dataset.pickle
and place it in the same folder as this notebook.

In [6]:
# Load the pickle file
with open('News_dataset.pickle', 'rb') as f:
    data = pickle.load(f)

# Convert to DataFrame
if isinstance(data, pd.DataFrame):
    df = data
elif isinstance(data, dict):
    df = pd.DataFrame(data)
else:
    df = pd.DataFrame(data)

print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

Dataset loaded successfully!
Shape: (2225, 6)
Columns: ['File_Name', 'Content', 'Category', 'Complete_Filename', 'id', 'News_length']


In [7]:
# Preview dataset
df.head()

,File_Name,Content,Category,Complete_Filename,id,News_length
0,001.txt,Ad sales boost Time Warner profit\r\n\r\nQuart...,business,001.txt-business,1,2569
1,002.txt,Dollar gains on Greenspan speech\r\n\r\nThe do...,business,002.txt-business,1,2257
2,003.txt,Yukos unit buyer faces loan claim\r\n\r\nThe o...,business,003.txt-business,1,1557
3,004.txt,High fuel prices hit BA's profits\r\n\r\nBriti...,business,004.txt-business,1,2421
4,005.txt,Pernod takeover talk lifts Domecq\r\n\r\nShare...,business,005.txt-business,1,1575


In [8]:
# Check column types and null values
print('Data Types:')
print(df.dtypes)
print('\nNull Values:')
print(df.isnull().sum())
print('\nCategory Distribution:')
# Try common column names for label
label_col = None
for col in ['category', 'label', 'class', 'Category', 'Label', 'Class', 'topic', 'Topic']:
    if col in df.columns:
        label_col = col
        break
if label_col is None:
    label_col = df.columns[-1]  # fallback: last column

text_col = None
for col in ['text', 'content', 'article', 'news', 'Text', 'Content', 'Article', 'News', 'headline']:
    if col in df.columns:
        text_col = col
        break
if text_col is None:
    text_col = df.columns[0]  # fallback: first column

print(f'\nUsing text column  : "{text_col}"')
print(f'Using label column : "{label_col}"')
print(f'\nLabel distribution:')
print(df[label_col].value_counts())

Data Types:
File_Name            object
Content              object
Category             object
Complete_Filename    object
id                    int64
News_length           int64
dtype: object

Null Values:
File_Name            0
Content              0
Category             0
Complete_Filename    0
id                   0
News_length          0
dtype: int64

Category Distribution:

Using text column  : "Content"
Using label column : "Category"

Label distribution:
Category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


## Step 3 — Text Cleaning
Removing HTML tags, URLs, special characters, numbers, and extra spaces.

In [9]:
def clean_text(text):
    """Full text cleaning pipeline."""
    if not isinstance(text, str):
        text = str(text)
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # 4. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 5. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # 6. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning
df['cleaned_text'] = df[text_col].apply(clean_text)

print('Text cleaning done!')
print('\n--- Before Cleaning ---')
print(df[text_col].iloc[0][:300])
print('\n--- After Cleaning ---')
print(df['cleaned_text'].iloc[0][:300])

Text cleaning done!

--- Before Cleaning ---
Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (Â£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and 

--- After Cleaning ---
ad sales boost time warner profit quarterly profits at us media giant timewarner jumped to bn â£m for the three months to december from m yearearlier the firm which is now one of the biggest investors in google benefited from sales of highspeed internet connections and higher advert sales timewarner


In [10]:
# Cleaning comparison table for first 3 rows
comparison = pd.DataFrame({
    'Original (first 100 chars)' : df[text_col].str[:100],
    'Cleaned (first 100 chars)'  : df['cleaned_text'].str[:100]
})
comparison.head(3)

,Original (first 100 chars),Cleaned (first 100 chars)
0,Ad sales boost Time Warner profit\r\n\r\nQuart...,ad sales boost time warner profit quarterly pr...
1,Dollar gains on Greenspan speech\r\n\r\nThe do...,dollar gains on greenspan speech the dollar ha...
2,Yukos unit buyer faces loan claim\r\n\r\nThe o...,yukos unit buyer faces loan claim the owners o...


## Step 4 — Remove Stop Words
Stop words are common words (the, is, at, which) that carry little meaning.

In [ ]:
stop_words = set(stopwords.words('english'))

print(f'Total stop words in NLTK: {len(stop_words)}')
print(f'Sample stop words: {list(stop_words)[:20]}')

def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words and len(word) > 1]
    return ' '.join(filtered)

df['no_stopwords'] = df['cleaned_text'].apply(remove_stopwords)

print('\nStop word removal done!')
print('\n--- After Cleaning ---')
print(df['cleaned_text'].iloc[0][:200])
print('\n--- After Stop Word Removal ---')
print(df['no_stopwords'].iloc[0][:200])

Total stop words in NLTK: 198
Sample stop words: ['down', 'at', "he's", "i'll", 'hasn', 'under', 'all', 'our', 'wasn', 'we', "she's", 'them', 're', 'there', 'from', "that'll", 'myself', 'did', 'your', 'until']


In [ ]:
# Show word count reduction
df['word_count_before'] = df['cleaned_text'].apply(lambda x: len(x.split()))
df['word_count_after']  = df['no_stopwords'].apply(lambda x: len(x.split()))
df['words_removed']     = df['word_count_before'] - df['word_count_after']

stats = pd.DataFrame({
    'Metric'   : ['Avg words before', 'Avg words after', 'Avg words removed', 'Reduction %'],
    'Value'    : [
        round(df['word_count_before'].mean(), 2),
        round(df['word_count_after'].mean(), 2),
        round(df['words_removed'].mean(), 2),
        round((df['words_removed'].mean() / df['word_count_before'].mean()) * 100, 2)
    ]
})
print(stats.to_string(index=False))

## Step 5 — Lemmatization
Using WordNet Lemmatizer to reduce words to their dictionary base form.

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(word, pos='v') for word in tokens]
    return ' '.join(lemmatized)

df['lemmatized_text'] = df['no_stopwords'].apply(lemmatize_text)

print('Lemmatization done!')
print('\n--- After Stop Word Removal ---')
print(df['no_stopwords'].iloc[0][:200])
print('\n--- After Lemmatization ---')
print(df['lemmatized_text'].iloc[0][:200])

In [ ]:
# Full preprocessing pipeline comparison
pipeline_df = pd.DataFrame({
    'Stage'  : ['Original', 'After Cleaning', 'After Stop Word Removal', 'After Lemmatization'],
    'Sample (first 120 chars)' : [
        df[text_col].iloc[0][:120],
        df['cleaned_text'].iloc[0][:120],
        df['no_stopwords'].iloc[0][:120],
        df['lemmatized_text'].iloc[0][:120]
    ]
})
pipeline_df

## Step 6 — Label Encoding
Converting text category labels into numeric values using `LabelEncoder`.

In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[label_col])

print('Label Encoding done!')
print(f'\nClasses found: {list(le.classes_)}')
print(f'Encoded as   : {list(range(len(le.classes_)))}')

# Mapping table
mapping_df = pd.DataFrame({
    'Category (Original)': le.classes_,
    'Encoded Label'      : range(len(le.classes_))
})
print('\nLabel Encoding Mapping:')
print(mapping_df.to_string(index=False))

In [ ]:
# Show original vs encoded
label_compare = df[[label_col, 'label_encoded']].drop_duplicates().sort_values('label_encoded')
label_compare.columns = ['Original Label', 'Encoded Value']
label_compare.reset_index(drop=True)

In [ ]:
# Distribution of encoded labels
print('Encoded Label Distribution:')
print(df['label_encoded'].value_counts().sort_index())

## Step 7 — TF-IDF Representation

In [ ]:
# Build TF-IDF on the fully preprocessed text
tfidf = TfidfVectorizer(
    max_features  = 5000,   # top 5000 most important words
    ngram_range   = (1, 2), # unigrams and bigrams
    min_df        = 2,      # ignore terms appearing in fewer than 2 docs
    sublinear_tf  = True    # apply log normalization to TF
)

X_tfidf = tfidf.fit_transform(df['lemmatized_text'])
y       = df['label_encoded'].values

print(f'TF-IDF matrix shape  : {X_tfidf.shape}')
print(f'Number of features   : {len(tfidf.vocabulary_)}')
print(f'Number of labels     : {len(le.classes_)}')
print(f'Sample features      : {tfidf.get_feature_names_out()[:20].tolist()}')

In [ ]:
# Display TF-IDF matrix as DataFrame (first 5 rows, first 15 features)
tfidf_df = pd.DataFrame(
    X_tfidf.toarray()[:5, :15],
    columns=tfidf.get_feature_names_out()[:15]
).round(4)

print('TF-IDF Matrix (first 5 rows, first 15 features):')
tfidf_df

In [ ]:
# Top 10 TF-IDF features per category
print('Top 10 TF-IDF Features per Category:\n')
feature_names = tfidf.get_feature_names_out()

for cat_idx, cat_name in enumerate(le.classes_):
    # Get all docs for this category
    cat_mask = (y == cat_idx)
    if cat_mask.sum() == 0:
        continue
    cat_tfidf = X_tfidf[cat_mask].toarray().mean(axis=0)
    top_indices = cat_tfidf.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    print(f'  [{cat_name}]: {top_words}')

## Step 8 — Save Outputs

In [ ]:
import pickle as pkl
import scipy.sparse

# 1. Save full preprocessed DataFrame as CSV
df.to_csv('preprocessed_news.csv', index=False)
print('Saved: preprocessed_news.csv')

# 2. Save TF-IDF matrix (sparse format)
scipy.sparse.save_npz('tfidf_matrix.npz', X_tfidf)
print('Saved: tfidf_matrix.npz')

# 3. Save TF-IDF vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pkl.dump(tfidf, f)
print('Saved: tfidf_vectorizer.pkl')

# 4. Save Label Encoder
with open('label_encoder.pkl', 'wb') as f:
    pkl.dump(le, f)
print('Saved: label_encoder.pkl')

# 5. Save encoded labels as numpy array
np.save('encoded_labels.npy', y)
print('Saved: encoded_labels.npy')

print('\nAll outputs saved successfully!')

In [ ]:
# Verify saved files by loading them back
loaded_df      = pd.read_csv('preprocessed_news.csv')
loaded_tfidf   = scipy.sparse.load_npz('tfidf_matrix.npz')
loaded_labels  = np.load('encoded_labels.npy')

print('Verification of saved files:')
print(f'  preprocessed_news.csv shape : {loaded_df.shape}')
print(f'  tfidf_matrix.npz shape      : {loaded_tfidf.shape}')
print(f'  encoded_labels.npy shape    : {loaded_labels.shape}')
print('\nAll files verified!')

In [ ]:
# Final summary
summary = pd.DataFrame({
    'Step'  : ['1. Load Data', '2. Text Cleaning', '3. Stop Word Removal',
               '4. Lemmatization', '5. Label Encoding', '6. TF-IDF'],
    'Output': [
        f'{df.shape[0]} news articles loaded',
        'Lowercase, remove HTML/URLs/numbers/punctuation',
        f'Removed NLTK stop words, avg {round(df["words_removed"].mean(),1)} words/doc',
        'WordNet lemmatizer (verb POS)',
        f'{len(le.classes_)} categories encoded as 0 to {len(le.classes_)-1}',
        f'Matrix: {X_tfidf.shape}, {len(tfidf.vocabulary_)} features'
    ]
})
print('=== Preprocessing Pipeline Summary ===')
print(summary.to_string(index=False))